In [1]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sys
sys.path.append('../')
import Modules_utility as util
import geopandas as gpd
from shapely.geometry import Point
import cartopy.crs as ccrs
import os

In [2]:
mdir="/orcd/nese/mhowland/001/lyqiu/GODEEP/"

# Land-use Filter

In [28]:
# regional data
ISO='ISONE'
national_restriction=xr.open_dataset(f'/orcd/nese/mhowland/001/lyqiu/NREL_Sitingmap/wind_coe_composite_50_{ISO}.nc')
regional_data_masked=xr.open_dataset(f'{mdir}/ISOs/{ISO}_highres_remap_landuse.nc')['mask']
regional_data_maskeda=xr.where(regional_data_masked>0.2,regional_data_masked,np.nan) # at least 2km2 area
regional_data_maskeda.to_netcdf(f'{mdir}/ISOs/{ISO}_highres_remap_landuse_masked.nc')

# get locations per county

In [32]:
#!/usr/bin/env python3
import xarray as xr
import numpy as np
import pandas as pd
import os
import argparse
import warnings
from geopy.distance import geodesic
warnings.filterwarnings('ignore')

mdir = "/orcd/nese/mhowland/001/lyqiu/GODEEP/NE_2025/preprocess/Network/"
ISO = 'ERCOT'
output_dir = "/orcd/nese/mhowland/001/lyqiu/GODEEP/NE_2025/preprocess/Resource/"

countylist = pd.read_csv(f"{mdir}/{ISO}_county_list.csv", dtype={'FIPS': int},index_col='FIPS')
countylist=countylist.drop(columns=['index','area_km2'])

# Load land use mask
mask = xr.open_dataset(f'{mdir}/{ISO}_landuse_masked.nc')['mask']
print(f"Loaded land use mask with shape: {mask.shape}")    

summary_data=[]
all_selected_locations=[]
for sta in countylist.index:
    county_info = countylist.loc[sta]
    maskfile = xr.open_dataset("/orcd/nese/mhowland/001/lyqiu/GODEEP/ISO_county/%s/remap_netcdf/%05d_remap.nc" % (ISO, sta))['mask']
    mask_county=maskfile*mask
    mask_county=mask_county.where(mask_county>0, np.nan)
    mask_county_stack= mask_county.stack(z=('y', 'x')).dropna('z')
    n_original_locations = (mask_county_stack>0).sum()
    
    if n_original_locations == 0:
        summary_data.append({
            'FIPS': sta,
            'Selected_Locations': 0,
        })
    else:
        selected_locations=mask_county_stack.to_dataframe().reset_index(drop=True)[['lon','lat','mask']]
        selected_locations['FIPS'] = sta
        selected_locations['distance_miles']=selected_locations.apply(lambda row: geodesic((county_info['lat'], county_info['lon']),(float(row['lat']),float(row['lon']))).miles, axis=1)
        n_selected_locations = len(selected_locations)
        if n_selected_locations>8:
            selected_locations=selected_locations.nlargest(8,'mask')

        all_selected_locations.append(selected_locations)
        
        summary_data.append({
            'FIPS': sta,
            'Selected_Locations': len(selected_locations)
        })

# # Create summary DataFrame
summary_df = pd.DataFrame(summary_data)
summary_df=pd.merge(summary_df,countylist,left_on='FIPS',right_index=True)
summary_df['Zone']=np.arange(1, len(summary_df) + 1).tolist()
summary_file = f"{output_dir}/candidates_{ISO}_summary.csv"
summary_df.to_csv(summary_file, index=False)


#Create selected locations DataFrame
locations_file = f"{output_dir}/candidates_{ISO}_by_county.csv"
selected_locations_df = pd.concat(all_selected_locations, ignore_index=False)
selected_locations_df=selected_locations_df.reset_index().rename(columns={'index':'loc_index'})
selected_locations_df.to_csv(locations_file, index=False)
print(len(selected_locations_df))



Loaded land use mask with shape: (95, 90)
1430


# Arrange CF data

In [5]:
model_input_dir = "./Resource/"
climate_scenario = 'historic'
iso_region = 'ISONE'
top_n=8
# Climate scenario selection
if climate_scenario == 'historic':
    full_year_list = list(range(2001, 2002))  # Historical period: 2001-2019
else:
    full_year_list = list(range(2020, 2021))  # Future period: 2020-2029

output_dir=f'./Resource/'


# =============================================================================
# DATA LOADING AND INITIALIZATION
# ============================================================================
countylist = pd.read_csv(
    f"{model_input_dir}/candidates_{iso_region}_summary.csv", 
    dtype={'FIPS': int},
)
countylist['Zone'] = np.arange(1, len(countylist) + 1).tolist()
countylist = countylist.set_index(['Zone'])

# Load detailed location data for renewable energy sites
location_data = pd.read_csv(
    f"{model_input_dir}/candidates_{iso_region}_by_county.csv", 
    dtype={'FIPS': int}
)

# =============================================================================
# MAIN PROCESSING LOOP: COUNTY-LEVEL RESOURCE SYNTHESIS
# =============================================================================
for technology_type in ['solar']:
    filesuffix = "/orcd/nese/mhowland/001/lyqiu/GODEEP/data/%s/%s/%s/%s_gen_cf_" % ( climate_scenario, technology_type, iso_region, technology_type)
    for year in full_year_list:
        print(f"Processing {technology_type} capacity factors for year {year}...")
        CF = xr.open_dataset(filesuffix + str(year) + '.nc')['capacity_factor']
        CF = CF.stack(z=('y', 'x')).dropna(dim='z')
        dates = CF['Time'].values.astype(str)
        
        try:
            base_dates = dates.astype(float).astype(int)  # e.g., 20200101
            frac_days = dates.astype(float) - base_dates  # e.g., 0.04166667
            base_datetimes = pd.to_datetime(base_dates, format="%Y%m%d")
            final_times = base_datetimes + pd.to_timedelta(frac_days, unit='D')
            final_times = final_times.round('h')
        except:
            final_times = pd.to_datetime(dates)
        CF['Time'] = final_times
        CF_df = CF.to_dataframe()[['capacity_factor', 'lat', 'lon']].reset_index()
        CF_df=CF_df.set_index(['lat','lon'])

        data_to_combine = {}
        if year==full_year_list[0]: 
            Vre_table=pd.DataFrame({'name':[], 'mask':[],'lat':[],'lon':[],'loc_index':[],'FIPS':[]})
        
        for zone_id in countylist.index:
            county_row = countylist.loc[zone_id]
            num_selected_locations = county_row['Selected_Locations']
            county_id = county_row['FIPS']
            selected_locations = location_data[location_data['FIPS'] == county_id]
            if (len(selected_locations) != num_selected_locations):
                print(f"Warning: Number of selected locations ({len(selected_locations)}) does not match summary ({num_selected_locations}) for county {county_id}")
            selected_locations=selected_locations.sort_values(by='loc_index').reset_index(drop=True)
            for i,location in selected_locations.iterrows():
                col_name = f"{county_id}_{technology_type}_{i}"
                capacity_factors = CF_df.loc[(location['lat'], location['lon']),'capacity_factor'].values
                data_to_combine[col_name] = capacity_factors
                vre_item=pd.DataFrame({'name': col_name, 
                                        'mask': location['mask']*144,
                                        'lat': location['lat'],
                                        'lon': location['lon'],
                                        'loc_index': int(i),
                                        'FIPS': county_id}, index=[0])
                if year==full_year_list[0]: 
                    Vre_table=pd.concat([Vre_table,vre_item],ignore_index=True)
            if num_selected_locations<top_n:
                for i in range(num_selected_locations, top_n):
                    data_to_combine[f"{county_id}_{technology_type}_{i}"] = 0
                    vre_item=pd.DataFrame({'name': f"{county_id}_{technology_type}_{i}", 
                                              'mask': 0,
                                              'lat': np.nan,
                                              'lon': np.nan,
                                              'loc_index': int(i),
                                              'FIPS': county_id}, index=[0])
                    if year==full_year_list[0]: 
                        Vre_table=pd.concat([Vre_table,vre_item],ignore_index=True)  
        power_DF = pd.DataFrame(data_to_combine)
        power_DF.index.name='Time'
        power_DF.to_csv(f'./Resource/{iso_region}/cf_{technology_type}/{climate_scenario}/{year}.csv')
        if year==full_year_list[0]:
            Vre_table.to_csv(f'./Resource/{iso_region}/cf_{technology_type}/Loc_table.csv')

Processing solar capacity factors for year 2001...


/tmp/ipykernel_1576768/1091505985.py:67: PerformanceWarning: indexing past lexsort depth may impact performance.
  capacity_factors = CF_df.loc[(location['lat'], location['lon']),'capacity_factor'].values


In [34]:
power_DF.to_csv(f'./Resource/{iso_region}/cf_{technology_type}/{climate_scenario}/{year}.csv')
if year==full_year_list[0]:
    Vre_table.to_csv(f'./Resource/{iso_region}/cf_{technology_type}/Loc_table.csv')

In [29]:
Vre_table

,name,mask,lat,lon,loc_index,FIPS
0,9001_solar_0,0.000000,NaN,NaN,0.0,9001.0
1,9001_solar_1,0.000000,NaN,NaN,1.0,9001.0
2,9001_solar_2,0.000000,NaN,NaN,2.0,9001.0
3,9001_solar_3,0.000000,NaN,NaN,3.0,9001.0
4,9001_solar_4,0.000000,NaN,NaN,4.0,9001.0
5,9001_solar_5,0.000000,NaN,NaN,5.0,9001.0
6,9001_solar_6,0.000000,NaN,NaN,6.0,9001.0
7,9001_solar_7,0.000000,NaN,NaN,7.0,9001.0
8,9003_solar_0,41.649110,41.969,-73.015,0.0,9003.0
9,9003_solar_1,0.000000,NaN,NaN,1.0,9003.0


# Arrange demand data

In [171]:
model_input_dir = "./Resource/"
climate_scenario = 'rcp45hotter'
iso_region = 'ISONE'

countylist = pd.read_csv(
    f"{model_input_dir}/candidates_{iso_region}_summary.csv", 
    dtype={'FIPS': int},
)
countylist['Zone'] = np.arange(1, len(countylist) + 1).tolist()
countylist = countylist.set_index(['Zone'])


# Climate scenario selection
if climate_scenario == 'historic':
    full_year_list = list(range(2001, 2020))  # Historical period: 2001-2019
else:
    full_year_list = list(range(2020, 2060))  # Future period: 2020-2059

output_dir=f'./Resource/'
DF_demand=pd.read_csv("/orcd/nese/mhowland/001/lyqiu/GODEEP/Demand/Demand_TELL/outputs/mlp_output/%s_%s_demand.csv"%(iso_region,climate_scenario),index_col=0)
DF_demand['Time_UTC']=pd.to_datetime(DF_demand['Time_UTC'])
DF_demand=DF_demand.set_index('Time_UTC')
demand_county=pd.DataFrame()
countylist['population_weight']=countylist['population']/countylist['population'].sum()
for zone_id in countylist.index:
    weight=countylist.loc[zone_id,'population_weight']
    a=DF_demand[['Load']]*weight
    a = a.rename(columns={'Load': zone_id})
    demand_county = pd.concat([demand_county, a], axis=1)
demand_county=demand_county.round(2)
demand_county = demand_county[~((demand_county.index.month == 2) & (demand_county.index.day == 29))]
demand_county.columns = [f"Demand_MW_z{col}" for col in demand_county.columns]

for year in full_year_list:
    demand_year = demand_county[demand_county.index.year==year].copy()
    try:
        demand_year['Time'] = range(8760)
    except:
        if len(demand_year) < 8760:
            demand_year = demand_year.reindex(range(8760), fill_value=0)
            demand_year['Time'] = range(8760)
    demand_year = demand_year.set_index('Time')
    demand_year.to_csv((f'{output_dir}/{iso_region}/demand/{climate_scenario}/{year}.csv'), index=True)
    
